# Part 4 — Vector DB: Embeddings & Similarity Demo
**Topics:** Cricket, Cooking, Cybersecurity  
**Model:** `all-MiniLM-L6-v2` via `sentence-transformers`

In [ ]:
# Install required libraries (run once in Colab)
!pip install sentence-transformers scikit-learn seaborn matplotlib -q

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print('Libraries loaded successfully.')

 ## Define 10 sentences across 3 topics

In [ ]:
sentences = [
    # Cricket (4 sentences)
    "The batsman hit a massive six over the boundary in the final over.",          # 0
    "India won the Test match after a brilliant century by the opener.",           # 1
    "The spinner deceived the batsman with a well-disguised googly.",              # 2
    "The fielder took a stunning catch at deep mid-wicket to end the innings.",   # 3

    # Cooking (3 sentences)
    "Sauté the onions in olive oil until they turn golden and translucent.",      # 4
    "Add the spices to the pan and let them bloom for thirty seconds.",           # 5
    "The secret to a perfect risotto is adding warm stock one ladle at a time.",  # 6

    # Cybersecurity (3 sentences)
    "The attacker used a phishing email to steal the employee's login credentials.",  # 7
    "Always enable two-factor authentication to protect your online accounts.",        # 8
    "A SQL injection attack can expose an entire database if inputs are not sanitised.",# 9
]

labels = [
    'C1: Six over boundary',
    'C2: India Test century',
    'C3: Spinner googly',
    'C4: Stunning catch',
    'K1: Sauté onions',
    'K2: Bloom spices',
    'K3: Risotto stock',
    'S1: Phishing email',
    'S2: Two-factor auth',
    'S3: SQL injection',
]

print(f'Total sentences: {len(sentences)}')
for i, s in enumerate(sentences):
    print(f'  [{i}] {s}')

##  — Load model and generate embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded:', model)

embeddings = model.encode(sentences, show_progress_bar=True)
print(f'Embedding shape: {embeddings.shape}')  # Expected: (10, 384)

## 3 — Compute 10×10 cosine similarity matrix

In [ ]:
similarity_matrix = cosine_similarity(embeddings)
print('Similarity matrix shape:', similarity_matrix.shape)
print(np.round(similarity_matrix, 3))

##  — Display 10×10 heatmap

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(
    similarity_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='white'
)
plt.title('10×10 Cosine Similarity Matrix — Cricket, Cooking, Cybersecurity', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150)
plt.show()
print('Heatmap saved as similarity_heatmap.png')

##  — Find top 2 most similar sentences to the query

In [ ]:
query = "The bowler took three wickets in one over."
print(f'Query: "{query}"\n')

# Encode the query using the same model
query_embedding = model.encode([query])

# Compute cosine similarity between query and all 10 sentences
query_similarities = cosine_similarity(query_embedding, embeddings)[0]

# Rank all 10 sentences by similarity score (descending)
ranked_indices = np.argsort(query_similarities)[::-1]

print('Top 2 most similar sentences:\n')
for rank, idx in enumerate(ranked_indices[:2], start=1):
    print(f'  Rank {rank} (score: {query_similarities[idx]:.4f})')
    print(f'  [{idx}] {sentences[idx]}')
    print()

print('All similarity scores:')
for idx in ranked_indices:
    print(f'  [{idx}] {query_similarities[idx]:.4f}  {sentences[idx][:60]}...')